In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
select * from ext_novartis.raw.hco limit 20

In [0]:
%sql
select * from ext_novartis.raw.hcp limit 20

In [0]:
%sql
select * from ext_novartis.raw.hcp_hco_relationship limit 20

In [0]:
df=spark.sql("select * from ext_novartis.raw.hcp_hco_relationship")
from pyspark.sql.functions import lit
df1=df.withColumn("end_date",to_date(lit("2099-12-31"),"yyyy-MM-dd"))
df1.write.format("delta").mode("overwrite").option("mergeSchema", "true").option("overwriteSchema", "true").saveAsTable("ext_novartis.raw.hcp_hco_relationship")


In [0]:
%sql
select * from ext_novartis.raw.hcp_hco_relationship

#### SCD2 implemetation

In [0]:
from pyspark.sql import Row

updates = [
    Row(doctor_id="D4", hospital_id="H26089", start_date="2023-06-01"),
    Row(doctor_id="D3", hospital_id="H7777", start_date="2023-06-01"),
    Row(doctor_id="D6", hospital_id="H5555", start_date="2023-06-01"),
    Row(doctor_id="D8", hospital_id="H8888", start_date="2023-06-01")
]

updates_df = spark.createDataFrame(updates)

s_df=updates_df.withColumn("end_date",to_date(lit("2099-12-31"),"yyyy-MM-dd"))\
    .withColumn("Iscurrent",lit('true'))
s_df.show()

In [0]:
from pyspark.sql import Row
from delta.tables import DeltaTable

t_table=DeltaTable.forName(spark,"ext_novartis.raw.hcp_hco_relationship")

t_table.alias("t").merge(s_df.alias("s"),
                         "t.doctor_id=s.doctor_id and t.Iscurrent=true")\
                        .whenMatchedUpdate(
                            set={
                                "end_date":to_date(current_timestamp(),"yyyy-MM-dd"),
                                "Iscurrent":lit("false")
                            }
                        )\
                        .whenNotMatchedInsert(
                            values={
                                "doctor_id": "s.doctor_id",
                                "hospital_id": "s.hospital_id",
                                "start_date": "s.start_date",
                                "end_date": "s.end_date",
                                "Iscurrent": "s.Iscurrent"
                            }
                        ).execute()

In [0]:
# Create staging rows
expire_rows = s_df.withColumn("mergeKey", col("doctor_id"))
insert_rows = s_df.withColumn("mergeKey", lit(None))

staged_df = expire_rows.union(insert_rows)
staged_df.show()

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

t_table = DeltaTable.forName(spark, "ext_novartis.raw.hcp_hco_relationship")

t_table.alias("t").merge(
    staged_df.alias("s"),
    "t.doctor_id = s.mergeKey AND t.Iscurrent = true"
)\
.whenMatchedUpdate(
    set={
        "end_date": "current_date()",
        "Iscurrent": "false"
    }
)\
.whenNotMatchedInsert(
    values={
        "doctor_id": "s.doctor_id",
        "hospital_id": "s.hospital_id",
        "start_date": "s.start_date",
        "end_date": "to_date('2099-12-31')",
        "Iscurrent": "true"
    }
)\
.execute()

In [0]:
%sql
select * from ext_novartis.raw.hcp_hco_relationship where doctor_id='D4'

In [0]:
%sql
select * from ext_novartis.raw.calls limit 20

In [0]:
%sql
select * from ext_novartis.raw.emails limit 20

In [0]:
%sql
select * from ext_novartis.raw.ad_impressions limit 20


In [0]:
%sql
select * from ext_novartis.raw.late_events limit 20

In [0]:
%sql
select * from ext_novartis.raw.cdc_calls limit 20

In [0]:
%sql
select * from ext_novartis.raw.bad_data limit 20

In [0]:
%sql
select * from ext_novartis.raw.campaigns limit 200
